In [1]:
from pathlib import Path

print(Path.cwd())

/Users/katrinarnadottir/Downloads/ALG_Group_Project


In [2]:
import numpy as np
import json

from pathlib import Path
from scipy.spatial.distance import cdist
from time import perf_counter
from tqdm.auto import tqdm

DATA_DIR = Path("wikipedia_nn_data")
K = 10

# Load embeddings
X = np.load(DATA_DIR / "embeddings.npy")
Q = np.load(DATA_DIR / "queries.npy")

# Load original sentences
with open(DATA_DIR / "sentences.json") as f:
    database_sentences = json.load(f)

with open(DATA_DIR / "query_sentences.json") as f:
    query_sentences = json.load(f)

print("Database shape:", X.shape)
print("Query shape:", Q.shape)

Database shape: (50000, 384)
Query shape: (500, 384)


In [3]:
def brute_force_search(X, q, k=10):

    # Compute squared Euclidean distances
    distances = cdist(
        q.reshape(1, -1),
        X,
        metric="sqeuclidean"
    ).ravel()

    # Select the k closest vectors
    indices = np.argpartition(distances, k - 1)[:k]

    # Sort selected neighbors by distance
    indices = indices[np.argsort(distances[indices])]

    return indices

In [4]:
i = 0

neighbors = brute_force_search(X, Q[i], K)

print("Query:")
print(query_sentences[i])

print("\nTop 10 nearest neighbors:")

for rank, idx in enumerate(neighbors, 1):
    print(f"{rank}. {database_sentences[idx]}")

Query:
It samples the Scott McKenzie song, "San Francisco (Be Sure to Wear Flowers in Your Hair)", and "California Dreamin'" by The Mamas & the Papas.

Top 10 nearest neighbors:
1. The song's lyrics are an ode to the city of San Francisco in Northern California.
2. In addition to Fall's score, it featuring songs by Jerome Kern and others, with additional lyrics Adrian Ross and others.
3. The song's beat contains samples from "You Make the Sun Shine" by The Temprees and "What It Is" by The Temptations.
4. The song plays during the opening and end credits.
5. The song is all about wet dreams filled with sexual innuendos and pop culture references.
6. It is based on the lyrics from "A Dagger Through the Heart of St. Angeles".
7. It is a sentimental song in the tradition of "Don't Sit Under the Apple Tree" or "I'll Be Seeing You".
8. The song was used in a Rimmel make up advert.
9. The song was written by Stefani and Williams and produced by The Neptunes.
10. The song has been featured in 

In [5]:
ground_truth = np.array([
    brute_force_search(X, q, K)
    for q in tqdm(Q, desc="Computing ground truth")
])

print("Ground truth shape:", ground_truth.shape)

Computing ground truth:   0%|          | 0/500 [00:00<?, ?it/s]

Ground truth shape: (500, 10)


In [6]:
np.save(DATA_DIR / "ground_truth.npy", ground_truth)

print("Ground truth saved successfully.")

Ground truth saved successfully.


In [7]:
REPEATS = 3
times = []

# Warm-up
brute_force_search(X, Q[0], K)

for _ in range(REPEATS):

    start = perf_counter()

    for q in Q:
        brute_force_search(X, q, K)

    elapsed = perf_counter() - start
    times.append(elapsed / len(Q))

brute_query_time = np.median(times)

print(f"Query time: {brute_query_time * 1000:.3f} ms")

Query time: 31.170 ms


In [8]:
from evaluation import evaluate_search

brute_results = evaluate_search(
    search_fn=lambda q, k: brute_force_search(X, q, k),
    Q=Q,
    ground_truth=ground_truth,
    k=10,
    repeats=3
)

print(brute_results)

{'recall': 1.0, 'query_time_ms': 34.07158474996686}


In [9]:
import pandas as pd

baseline_results = pd.DataFrame([
    {
        "algorithm": "Brute force",
        "n": X.shape[0],
        "dimension": X.shape[1],
        "k": K,
        "recall_at_10": brute_results["recall"],
        "query_time_ms": brute_results["query_time_ms"],
        "build_time_s": 0.0
    }
])

baseline_results.to_csv(
    DATA_DIR / "baseline_results.csv",
    index=False
)

baseline_results

,algorithm,n,dimension,k,recall_at_10,query_time_ms,build_time_s
0,Brute force,50000,384,10,1.0,34.071585,0.0
